In [1]:
import progtv
from datetime import datetime
from pathlib import Path
import pandas as pd

/home/david/miniconda3/envs/progtv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-02-23 21:40:54.034880: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-23 21:40:54.047546: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-23 21:40:54.051893: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-23 21:40:54.062795: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow 

In [2]:
tv_program = progtv.TVProgram()
current_directory = Path.cwd()
file_name_rated = f"{current_directory}/{tv_program.download_folder}/progtv_rated_{datetime.now().today().strftime('%Y-%m-%d')}.pkl"
rated_progs = tv_program.read_programs(file_name_rated)

In [3]:
rated_progs

,id,name,icon,programs
0,TF1.fr,TF1,https://www.teleboy.ch/assets/stations/308/ico...,...
1,France2.fr,France 2,https://www.teleboy.ch/assets/stations/342/ico...,name ...
2,France3.fr,France 3,https://www.teleboy.ch/assets/stations/58/icon...,...
3,CanalPlus.fr,Canal+,https://focus.telerama.fr/500x500/0000/00/01/c...,name ...
4,France5.fr,France 5,https://www.teleboy.ch/assets/stations/80/icon...,...
5,M6.fr,M6,https://www.teleboy.ch/assets/stations/312/ico...,name ...
6,Arte.fr,Arte,https://www.teleboy.ch/assets/stations/330/ico...,...
7,C8.fr,C8,https://focus.telerama.fr/500x500/0000/00/01/c...,name ...
8,W9.fr,W9,https://www.teleboy.ch/assets/stations/268/ico...,name ...
9,TMC.fr,TMC,https://www.teleboy.ch/assets/stations/383/ico...,name s...


In [11]:
rated_progs.iloc[0]["programs"].nlargest(5, "note_pred")

,name,start,end,channel,icon,rating,cat,desc,embeddings_camembert,cat_encoded,rating_encoded,note_pred
60,Prise au piège dans ma maison,2025-02-24 15:50:00,2025-02-24 17:35:00,TF1.fr,https://img.bouygtel.fr/CMS/images/0A07551B6A9...,-10,Action,"Kyle Hastings, créateur d'un projet technologi...","[[0.02052811, -0.07615078, 0.0009498594, 0.133...",0,0,0.409370
3,Petits plats en équilibre,2025-02-22 11:45:00,2025-02-22 11:50:00,TF1.fr,https://img.bouygtel.fr/CMS/images/C4F6912865B...,Tout public,Art de vivre,Laurent Mariotte donne toutes les astuces pour...,"[[0.08142108, 0.070667215, -0.03368971, 0.0144...",1,2,0.384256
5,Petits plats en équilibre,2025-02-22 12:50:00,2025-02-22 12:55:00,TF1.fr,https://img.bouygtel.fr/CMS/images/C4F6912865B...,Tout public,Art de vivre,Laurent Mariotte donne toutes les astuces pour...,"[[0.08142108, 0.070667215, -0.03368971, 0.0144...",1,2,0.384256
13,Petits plats en équilibre,2025-02-22 19:50:00,2025-02-22 19:55:00,TF1.fr,https://img.bouygtel.fr/CMS/images/C4F6912865B...,Tout public,Art de vivre,Laurent Mariotte donne toutes les astuces pour...,"[[0.08142108, 0.070667215, -0.03368971, 0.0144...",1,2,0.384256
28,Petits plats en équilibre,2025-02-23 12:50:00,2025-02-23 12:55:00,TF1.fr,https://img.bouygtel.fr/CMS/images/F4A4D49AED9...,Tout public,Art de vivre,Laurent Mariotte donne toutes les astuces pour...,"[[0.08142108, 0.070667215, -0.03368971, 0.0144...",1,2,0.384256


In [27]:
def get_best_programs(rated_progs, n=5, whitelist=None):
    best_progs = pd.DataFrame()
    best_progs_whitelist = pd.DataFrame()
    for i, row in rated_progs.iterrows():
        best_prog = row.programs.nlargest(n, "note_pred")
        # Calculate the duration of each program
        best_prog.loc[:, "duration"] = (best_prog["end"] - best_prog["start"]).dt.total_seconds() / 60
        # add channel name and icon
        try:
            best_prog.loc[:, "channel_name"] = row["name"]
            best_prog.loc[:, "channel_icon"] = row["icon"]
        except:
                pass
        best_progs = pd.concat([best_progs, best_prog])
        # drop duplicates programs
        best_progs = best_progs.drop_duplicates(subset=["name"])
        for channel in whitelist:
            best_prog_whitelist = best_progs[best_progs["channel_name"] == channel].nlargest(1, "note_pred")
            best_progs_whitelist = pd.concat([best_progs_whitelist, best_prog_whitelist])
        best_progs = best_progs.nlargest(n, "note_pred")
        best_progs = pd.concat([best_progs, best_progs_whitelist])
        best_progs = best_progs.drop_duplicates(subset=["name"])
    return best_progs

In [28]:
get_best_programs(rated_progs, n=5, whitelist=["TF1", "France 2", "France 3", "Canal+"])

,name,start,end,channel,icon,rating,cat,desc,embeddings_camembert,cat_encoded,rating_encoded,note_pred,duration,channel_name,channel_icon
112,J'irai dormir chez vous,2025-02-28 21:10:00,2025-02-28 23:00:00,RMCDecouverte.fr,https://img.bouygtel.fr/CMS/images/9CFA11B08A8...,Tout public,Aucun genre,Le plus célèbre des globetrotters prend cette ...,"[[0.0017619004, 0.02942601, 0.008365242, 0.104...",0,2,0.548742,110.0,RMC Découverte,https://www.teleboy.ch/assets/stations/380/ico...
3,Mayas : comment la NASA explique leur chute,2025-02-22 09:20:00,2025-02-22 10:40:00,RMCDecouverte.fr,https://img.bouygtel.fr/CMS/images/C4A744E7571...,Tout public,Aucun genre,Le changement climatique et la pollution serai...,"[[-0.00096423266, 0.0260814, -0.11004131, 0.10...",0,2,0.500850,80.0,RMC Découverte,https://www.teleboy.ch/assets/stations/380/ico...
103,Commissaire Magellan,2025-02-25 02:57:00,2025-02-25 06:21:00,C8.fr,https://img.bouygtel.fr/CMS/images/8C2B8ACC73C...,Tout public,Aucun genre,Lucie Mauricourt et Loréna Gardin sont deux di...,"[[0.024951749, 0.023332324, 0.038352206, 0.142...",0,1,0.490345,204.0,C8,https://focus.telerama.fr/500x500/0000/00/01/c...
4,Machu Picchu : les secrets de la cité perdue d...,2025-02-22 10:40:00,2025-02-22 11:45:00,RMCDecouverte.fr,https://img.bouygtel.fr/CMS/images/D27DDFAD57F...,Tout public,Aucun genre,"Dans les montagnes du sud du Pérou, le Machu P...","[[0.022821248, 0.069661975, -0.06457121, 0.058...",0,2,0.484559,65.0,RMC Découverte,https://www.teleboy.ch/assets/stations/380/ico...
20,Très très bon,2025-02-23 11:25:00,2025-02-23 12:00:00,ParisPremiere.fr,https://img.bouygtel.fr/CMS/images/13C61DA6925...,Tout public,Art de vivre,"Tables bistronomiques, adresses traditionnelle...","[[0.041976355, 0.14343691, 0.023603069, 0.0717...",1,3,0.481715,35.0,Paris Première,https://focus.telerama.fr/500x500/0000/00/01/c...
60,Prise au piège dans ma maison,2025-02-24 15:50:00,2025-02-24 17:35:00,TF1.fr,https://img.bouygtel.fr/CMS/images/0A07551B6A9...,-10,Action,"Kyle Hastings, créateur d'un projet technologi...","[[0.02052811, -0.07615078, 0.0009498594, 0.133...",0,0,0.409370,105.0,TF1,https://www.teleboy.ch/assets/stations/308/ico...
65,Flair de famille,2025-02-24 00:26:00,2025-02-24 01:50:00,France2.fr,https://img.bouygtel.fr/CMS/images/BB2DFDBB163...,Tout public,Aucun genre,"Quand le corps de Jackie Lefranc, ex ""Reine de...","[[0.033027608, 0.012889833, 0.004484705, 0.149...",1,1,0.408821,84.0,France 2,https://www.teleboy.ch/assets/stations/342/ico...
29,Outremer.gourmand,2025-02-22 10:35:00,2025-02-22 11:15:00,France3.fr,https://img.bouygtel.fr/CMS/images/AF15365CB66...,Tout public,Art de vivre,"Désormais aux commandes de ""La soirée continue...","[[0.057680957, 0.11863465, -0.021310246, 0.035...",0,1,0.415828,40.0,France 3,https://www.teleboy.ch/assets/stations/58/icon...
52,Le Successeur,2025-02-24 01:11:00,2025-02-24 03:00:00,CanalPlus.fr,https://img.bouygtel.fr/CMS/images/F48B0394B48...,-10,Action,Tout juste promu directeur artistique d'une ma...,"[[0.01249852, -0.063608825, 0.016820846, 0.125...",0,0,0.453401,109.0,Canal+,https://focus.telerama.fr/500x500/0000/00/01/c...
